# VoxMeet Kaggle worker v2 — Việt / English
Private session: multilingual ASR + Community-1 VBx, live ASR, Qwen3-4B summary/chat.
GPU and Internet required. Setup, restart if packages changed, then run worker. No training.
Live uses 6-second chunks and source labels, not speaker identities; 1–2s latency is not guaranteed.


In [ ]:
import sys, subprocess, os
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError, distributions

wanted = {'numpy':'2.2.6', 'pyannote.audio':'4.0.7', 'faster-whisper':'1.2.1'}
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None

if all(installed(k) == v for k,v in wanted.items()) and installed('transformers') and installed('accelerate'):
    print('Libraries ready. Continue below.')
else:
    pins = dict(wanted)
    for name in ('torch','torchaudio','torchvision'):
        if installed(name): pins[name] = installed(name)
    for dist in distributions():
        name = dist.metadata.get('Name', '').lower().replace('_','-')
        if name.startswith('opentelemetry-'): pins[name] = dist.version
    constraints = Path('/kaggle/working/voxmeet-demo-constraints.txt')
    constraints.write_text('\n'.join(f'{k}=={v}' for k,v in pins.items()))
    env = os.environ.copy()
    env.pop('PIP_CONSTRAINT', None)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(constraints),
                    *[f'{k}=={v}' for k,v in wanted.items()], 'jedi==0.19.2', 'transformers>=4.51,<6', 'accelerate>=1,<2'],
                   env=env, check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'check'], env=env, check=False)
    print('Setup complete. Run → Restart & clear cell outputs, then run the next cell.')


In [ ]:
import json, threading, time, traceback, textwrap
import requests
from kaggle_secrets import UserSecretsClient

WORKER_URL = WORKER_URL_PLACEHOLDER
WORKER_KEY = WORKER_KEY_PLACEHOLDER
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
HEADERS = {'Authorization': 'Bearer ' + WORKER_KEY}
BASE = WORKER_URL.split('/worker/')[0]
stop = threading.Event()

# Verify the environment BEFORE advertising availability or downloading audio.
print('Checking NumPy, pyannote and GPU before accepting jobs...', flush=True)
try:
    from importlib.metadata import version
    import numpy as np
    import numpy.strings
    import scipy.signal
    import torch
    from pyannote.audio import Pipeline
    assert np.__version__ == version('numpy'), 'Loaded NumPy differs from installed NumPy.'
    assert torch.cuda.is_available(), 'Enable Kaggle GPU accelerator first.'
    probe = torch.ones(2, device='cuda')
    torch.testing.assert_close(probe + probe, torch.full_like(probe, 2))
    del probe
except Exception as exc:
    raise RuntimeError(
        'Worker has NOT started. If setup installed packages, choose Run > Restart & clear cell outputs, '
        'then run ONLY this worker cell. Do not Run All. If imports still fail, send the preflight traceback.'
    ) from exc
print('Preflight OK:', np.__version__, '|', torch.cuda.get_device_name(0), flush=True)

COMMON_TURN_SOURCE = COMMON_TURN_SOURCE_PLACEHOLDER
TURN_ENGINE = {}
exec(COMMON_TURN_SOURCE, TURN_ENGINE)
SpeakerMemory = TURN_ENGINE['SpeakerMemory']

def heartbeat():
    while not stop.is_set():
        try:
            r = requests.post(WORKER_URL+'/heartbeat', headers=HEADERS, json={'features':['multilingual','llm','live_asr','live_speakers','shared_turns']+(['live_refine'] if refinement_ready.is_set() else [])}, timeout=15)
            if r.status_code in (401, 403, 410):
                stop.set()
        except requests.RequestException:
            pass
        stop.wait(20)

def acoustic_candidates(samples, units, pipeline):
    """One shared embedding extractor for Live and Record sentence/audio units."""
    candidates=[]
    with torch.inference_mode():
        for unit in units:
            a,b=float(unit['start']),float(unit['end'])
            candidate=dict(start=a,end=b,vector=None,purity=0.)
            if b-a>=.8:
                width=min(1.8,b-a)
                starts=sorted({a,(a+b-width)/2,b-width})
                vectors=[]
                for begin in starts:
                    clip=samples[max(0,round(begin*16000)):min(len(samples),round((begin+width)*16000))]
                    if len(clip)<12000:continue
                    value=np.asarray(pipeline._embedding(torch.from_numpy(clip[None,None,:])),dtype=np.float32).reshape(-1)
                    norm=np.linalg.norm(value)
                    if np.isfinite(value).all() and norm>0:vectors.append(value/norm)
                if vectors:
                    purity=min(float(np.dot(x,y)) for x in vectors for y in vectors)
                    if purity>=.55:
                        vector=np.mean(vectors,axis=0);norm=np.linalg.norm(vector)
                        if norm>0:candidate.update(vector=vector/norm,purity=purity)
            candidates.append(candidate)
    return candidates


def record_sentence_clustering(samples, words, turns, pipeline, max_speakers=8):
    units=TURN_ENGINE['record_sentences'](words,turns)
    candidates=acoustic_candidates(samples,units,pipeline)
    labels=TURN_ENGINE['cluster_record_embeddings'](candidates,max_speakers=max_speakers)
    result=[]
    for index,(unit,label,candidate) in enumerate(zip(units,labels,candidates),1):
        result.append(dict(id=f'U{index:04d}',start=unit['start'],end=unit['end'],text=unit['text'],
            speaker=label,words=[dict(w,speaker=label) for w in unit['words']],
            speaker_status='sentence_acoustic' if label else 'uncertain',
            acoustic_purity=round(candidate['purity'],3)))
    return result


def audio_embedding(samples, pipeline):
    with torch.inference_mode():
        return np.asarray(pipeline._embedding(torch.from_numpy(samples[None,None,:])),dtype=np.float32).reshape(-1)


class AudioEmbeddingBatch:
    """Exact-length GPU batches: no padding or waveform normalization changes."""
    def __init__(self,samples,pipeline,batch_size=16):
        self.samples=samples;self.pipeline=pipeline;self.batch_size=batch_size
        self.cache={};self.calls=0;self.seconds=0.;self.batch_verified=None
    def key(self,a,b):return (max(0,round(a*16000)),min(len(self.samples),round(b*16000)))
    def infer(self,keys):
        if not keys:return
        array=np.stack([self.samples[a:b] for a,b in keys])[:,None,:]
        try:
            with torch.inference_mode():
                values=np.asarray(self.pipeline._embedding(torch.from_numpy(array)),dtype=np.float32)
            self.calls+=1
        except RuntimeError as exc:
            if 'out of memory' not in str(exc).lower() or len(keys)==1:raise
            torch.cuda.empty_cache()
            half=max(1,len(keys)//2);self.infer(keys[:half]);self.infer(keys[half:]);return
        if values.ndim!=2 or len(values)!=len(keys):raise ValueError('Invalid batched speaker embeddings')
        for key,value in zip(keys,values):self.cache[key]=value
    def prepare(self,ranges):
        began=time.perf_counter();groups={}
        for a,b in ranges:
            key=self.key(a,b)
            if key[1]>key[0] and key not in self.cache:groups.setdefault(key[1]-key[0],set()).add(key)
        for length,keys in groups.items():
            keys=sorted(keys)
            start=0
            while start<len(keys):
                chunk=keys[start:start+self.batch_size];start+=len(chunk);self.infer(chunk)
                # Check the real installed embedding implementation, not just
                # tensor shape. Fall back to batch 1 if its output differs.
                if len(chunk)>1 and self.batch_verified is None:
                    batch_value=self.cache[chunk[0]].copy();self.infer(chunk[:1])
                    single=self.cache[chunk[0]]
                    norms=float(np.linalg.norm(single)*np.linalg.norm(batch_value))
                    self.batch_verified=bool(norms>0 and float(single@batch_value)/norms>=.999)
                    if not self.batch_verified:
                        self.batch_size=1
                        for key in chunk:self.infer([key])
                        print('Embedding batch consistency check failed; using exact single-window inference.',flush=True)
        self.seconds+=time.perf_counter()-began
    def at(self,a,b):
        key=self.key(a,b)
        if key not in self.cache:self.infer([key])
        return self.cache[key]


def verify_audio_tracks(samples, tracks, pipeline, include_vectors=False):
    embedding=AudioEmbeddingBatch(samples,pipeline)
    corrected,diagnostic=TURN_ENGINE['acoustic_resegment'](samples,tracks,embedding,include_vectors=include_vectors)
    diagnostic[0]['embedding_performance']=dict(windows=len(embedding.cache),calls=embedding.calls,
        seconds=round(embedding.seconds,3),batch_size=embedding.batch_size,batch_consistency=embedding.batch_verified)
    return corrected,diagnostic


record_pipeline=None

def get_record_pipeline(revision):
    global record_pipeline
    if record_pipeline is None:
        from huggingface_hub import snapshot_download
        from pyannote.audio import Pipeline
        model_dir=snapshot_download('pyannote/speaker-diarization-community-1',revision=revision,token=HF_TOKEN)
        record_pipeline=Pipeline.from_pretrained(model_dir).to(torch.device('cuda:0'))
    return record_pipeline


def refine_sentence_units(samples,units,tracks,session,offset):
    # Voice recognition is audio based. Transcript units are only display metadata.
    memory=refine_sessions.setdefault(session,SpeakerMemory())
    corrected,diagnostic=verify_audio_tracks(samples,tracks,refine_pipeline,include_vectors=True)
    candidates=[]
    for window in diagnostic:
        if 'embedding' not in window or window.get('kind')=='short_track':continue
        a,b=window['start'],window['end']
        vector=np.asarray(window['embedding'],dtype=np.float32)
        candidates.append((window,vector))
        memory.observe(vector,offset+a,offset+b)
    candidates.sort(key=lambda pair:((pair[0]['start']+pair[0]['end'])/2,pair[0].get('kind')=='short_track'))
    result=[]
    for index,(track,vector) in enumerate(candidates):
        # Assign disjoint time spans; overlapping analysis windows must not
        # create two competing identities at the same timestamp.
        center=(track['start']+track['end'])/2
        previous=candidates[index-1][0] if index else None
        following=candidates[index+1][0] if index+1<len(candidates) else None
        begin=(center+(previous['start']+previous['end'])/2)/2 if previous and previous['interval']==track['interval'] else track['start']
        finish=(center+(following['start']+following['end'])/2)/2 if following and following['interval']==track['interval'] else track['end']
        identity=memory.identify(vector)
        norm=float(np.linalg.norm(vector))
        scores=sorted([float((vector/norm)@p['vector']) for p in memory.prototypes if p['speaker']],reverse=True) if norm>0 else []
        score=scores[0] if scores else 0.
        margin=score-scores[1] if len(scores)>1 else score+1
        label=identity['speaker'] if identity else None
        result.append(dict(start=begin,end=finish,speaker=label,
            evidence=dict(policy='audio-reference-v2.1',accepted=bool(label),similarity=score,margin=margin,
                support_seconds=identity['seconds'] if identity else 0)))
    short=[]
    for window in diagnostic:
        if window.get('kind')!='short_track' or 'embedding' not in window:continue
        vector=np.asarray(window['embedding'],dtype=np.float32);norm=float(np.linalg.norm(vector))
        identity=memory.identify(vector)
        known=[p for p in memory.prototypes if p['speaker']]
        scores=sorted([float((vector/norm)@p['vector']) for p in known],reverse=True) if norm>0 else []
        if identity and len(scores)>=2 and scores[0]>=.60 and scores[0]-scores[1]>=.20:
            short.append(dict(start=window['start'],end=window['end'],speaker=identity['speaker'],
                evidence=dict(policy='audio-reference-v2.1',accepted=True,similarity=scores[0],
                    margin=scores[0]-scores[1],support_seconds=identity['seconds'])))
    return TURN_ENGINE['overlay_verified_tracks'](result,short)



def process(raw_zip):
    import gc, hashlib, io, json, time, zipfile
    from pathlib import Path
    import numpy as np
    import soundfile as sf
    import torch
    from huggingface_hub import snapshot_download
    from pyannote.audio import Pipeline
    from pyannote.audio.pipelines.utils.hook import ProgressHook

    assert torch.cuda.is_available(), 'Enable T4 GPU first.'
    with zipfile.ZipFile(io.BytesIO(raw_zip)) as z:
        assert set(z.namelist()) == {'audio.wav', 'job.json'}, 'Not a VoxMeet audio package.'
        assert z.getinfo('audio.wav').file_size <= 120 * 1024 * 1024
        assert z.getinfo('job.json').file_size < 10000
        job = json.loads(z.read('job.json'))
        raw_audio = z.read('audio.wav')
    assert job['schema'] == 'voxmeet-gpu-v1'
    assert hashlib.sha256(raw_audio).hexdigest() == job['audio_sha256']
    samples, sr = sf.read(io.BytesIO(raw_audio), dtype='float32', always_2d=True)
    assert sr == 16000 and samples.shape[1] == 1 and np.isfinite(samples).all()
    samples = samples[:, 0].copy()
    assert 0 < len(samples) / sr <= 1801
    revision = '3533c8cf8e369892e6b79ff1bf80f7b0286a54ee'
    assert job['model_revision'] == revision
    started = time.perf_counter()
    language = job.get('language', 'en')
    assert language in ('vi','en','auto')
    asr_started=time.perf_counter()
    recognized = recognize(samples, language, live=False)
    asr_seconds=time.perf_counter()-asr_started
    words = recognized['words']
    if torch.cuda.device_count()<2:release_asr()
    assert words, 'No speech detected. Try clearer audio; no transcript will be replaced.'
    print('Loading Community-1 baseline (worker audio-reference-v2.1)...', flush=True)
    pipeline = get_record_pipeline(revision)
    diarization_started=time.perf_counter()
    with ProgressHook() as hook:
        output = pipeline({'waveform':torch.from_numpy(samples[None,:]), 'sample_rate':sr}, hook=hook, **({'num_speakers': job['num_speakers']} if job.get('num_speakers') else {}))
    diarization_seconds=time.perf_counter()-diarization_started
    verification_started=time.perf_counter()
    turns = [dict(start=float(t.start), end=float(t.end), speaker=label)
             for t, _, label in output.speaker_diarization.itertracks(yield_label=True)]
    baseline_turns=turns
    speaker_diagnostic=[]
    if job.get('turn_policy')=='shared-sentence-v1':
        annotation=output.exclusive_speaker_diarization
        exclusive=[dict(start=float(t.start),end=float(t.end),speaker=label) for t,_,label in annotation.itertracks(yield_label=True)]
        turns,speaker_diagnostic=verify_audio_tracks(samples,exclusive,pipeline)
        info=speaker_diagnostic[0]
        print('Audio reference v2.1:',json.dumps(info,ensure_ascii=False),flush=True)
    verification_seconds=time.perf_counter()-verification_started
    from collections import defaultdict
    def overlap(a, b):
        return max(0, min(a['end'], b['end']) - max(a['start'], b['start']))

    def group_words(words):
        segments = []
        for word in words:
            text = word.get('punctuated_word', word.get('word', '')).strip()
            if not text:
                continue
            speaker = word.get('speaker')
            if speaker is not None:
                speaker = str(speaker)
            if (not segments or segments[-1]['speaker'] != speaker or
                    word['start'] - segments[-1]['end'] > 2.5 or
                    word['end'] - segments[-1]['start'] > 30):
                segments.append(dict(id=f's{len(segments)+1}', speaker=speaker,
                                     start=float(word['start']), end=float(word['end']), text=text, words=[]))
            else:
                segments[-1]['text'] += ' ' + text
                segments[-1]['end'] = max(segments[-1]['end'], float(word['end']))
            segments[-1]['words'].append(dict(text=text, start=float(word['start']), end=float(word['end']),
                                             confidence=word.get('confidence')))
        return segments

    def assign_speakers(words, turns):
        for word in words:
            # Whisper sometimes returns point timestamps for short words. Assign
            # only when exactly one speaker is active at that point.
            if word['start'] == word['end']:
                speakers = {turn['speaker'] for turn in turns if turn['start'] <= word['start'] < turn['end']}
                word['speaker'] = next(iter(speakers)) if len(speakers) == 1 else None
                continue
            # A voice can have several adjacent intervals within one word. Compare
            # total covered time per voice, merging intervals to avoid double count.
            intervals = defaultdict(list)
            for turn in turns:
                if overlap(word, turn) > 0:
                    intervals[turn['speaker']].append((max(word['start'], turn['start']),
                                                       min(word['end'], turn['end'])))
            scores = {}
            for speaker, spans in intervals.items():
                total, right = 0., float('-inf')
                for start, end in sorted(spans):
                    total += max(0., end - max(start, right))
                    right = max(right, end)
                scores[speaker] = total
            ranked = sorted(scores, key=scores.get, reverse=True)
            word['speaker'] = ranked[0] if ranked and (len(ranked) == 1 or
                scores[ranked[0]] - scores[ranked[1]] > 1e-6) else None
            word['ambiguous_speaker'] = len(ranked) > 1
        return words

    if job.get('turn_policy')=='shared-sentence-v1':
        utterances=TURN_ENGINE['baseline_sentence_labels'](words,turns)
        words=[dict(word,word=word.get('word',word.get('text',''))) for unit in utterances for word in unit['words']]
    else:
        utterances = group_words(assign_speakers(words, turns))
    for index, utterance in enumerate(utterances, 1):
        utterance['id'] = f'U{index:04d}'
    result = dict(job, processing_timing=dict(asr_seconds=round(asr_seconds,3),diarization_seconds=round(diarization_seconds,3),verification_seconds=round(verification_seconds,3)), utterances=utterances, words=words, diarization_turns=turns, baseline_diarization_turns=baseline_turns, speaker_diagnostic=speaker_diagnostic, provenance=dict(
        worker_build='audio-reference-v2.2',asr_model='Whisper large-v3-turbo',language=recognized['language'],clustering=pipeline.clustering.__class__.__name__,diarization_model='Community-1 baseline',
        speaker_verifier='audio-reference-v2.1' if job.get('turn_policy')=='shared-sentence-v1' else 'baseline',diarization_revision=revision,assignment_policy=job.get('turn_policy','legacy'),
        caption_core_sha256=hashlib.sha256(COMMON_TURN_SOURCE.encode()).hexdigest()), processing_seconds=round(time.perf_counter()-started,1))
    result_path = Path('/kaggle/working') / f"voxmeet-result-{job['recording_id']}.json"
    result_path.write_text(json.dumps(result,ensure_ascii=False), encoding='utf-8')
    print('Record timing:',result['processing_timing'],flush=True)
    print('Done:',len(utterances),'turns |',len({u['speaker'] for u in utterances if u.get('speaker') is not None}),'speaker groups')
    print('Unassigned words:',sum(w['speaker'] is None for w in words))
    print('Review speaker labels in the local demo before summarizing.')
    return result_path.read_bytes()

# Embedded in the private session notebook; no local GPU or LLM required.
import gc, io, os, subprocess, sys, zipfile
from pathlib import Path

asr_process = None
llm_model = None
llm_tokenizer = None


def release_asr():
    global asr_process
    if asr_process is not None:
        asr_process.terminate()
        try: asr_process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            asr_process.kill(); asr_process.wait()
        asr_process = None


def release_llm():
    global llm_model, llm_tokenizer
    llm_model = llm_tokenizer = None
    gc.collect(); torch.cuda.empty_cache()


ASR_SERVICE = r'''
import json, sys, time, traceback
from pathlib import Path
import numpy as np
from faster_whisper import WhisperModel
model = None
model_name = None
for line in sys.stdin:
    try:
        task = json.loads(line)
        selected = 'turbo'  # Same warmed model for live and saved recordings.
        if model_name != selected:
            model = None
            model = WhisperModel(selected,device='cuda',compute_type='float16')
            model_name = selected
        audio = np.load(task['path'])
        inference_started = time.perf_counter()
        parts, info = model.transcribe(audio,language=None if task['language']=='auto' else task['language'],
            task='transcribe',beam_size=1 if task['live'] else 5,
            word_timestamps=True,vad_filter=True,condition_on_previous_text=False)
        words=[];segments=[]
        for part in parts:
            # Silence filtering is a heuristic, not a guarantee against hallucination.
            if part.no_speech_prob > .6 and part.avg_logprob < -1: continue
            for w in part.words or []:
                words.append(dict(start=float(w.start),end=float(w.end),word=w.word,confidence=float(w.probability)))
                if task['live']:
                    segments.append(dict(start=float(w.start),end=float(w.end),text=w.word))
        result=dict(words=words,segments=segments,language=info.language,
            asr_model=model_name,inference_seconds=time.perf_counter()-inference_started)
        print(json.dumps(result,ensure_ascii=False),flush=True)
    except Exception:
        traceback.print_exc(file=sys.stderr)
        print(json.dumps({'error':'ASR failed'}),flush=True)
'''


def recognize(samples, language, live):
    global asr_process
    if torch.cuda.device_count() < 2:
        release_llm()
    work=Path('/kaggle/working/voxmeet_asr_job');work.mkdir(exist_ok=True)
    if asr_process is None or asr_process.poll() is not None:
        cuda_root=Path('/kaggle/working/voxmeet_cuda12')
        libs=('libcublas.so.12','libcublasLt.so.12','libcudnn.so.9','libcudart.so.12','libnvrtc.so.12')
        if not all(list(cuda_root.rglob(name)) for name in libs):
            env=os.environ.copy();env.pop('PIP_CONSTRAINT',None)
            subprocess.run([sys.executable,'-m','pip','install','--upgrade','--no-deps','--target',str(cuda_root),
                'nvidia-cublas-cu12','nvidia-cudnn-cu12>=9,<10','nvidia-cuda-runtime-cu12','nvidia-cuda-nvrtc-cu12'],env=env,check=True)
        assert all(list(cuda_root.rglob(name)) for name in libs)
        env=os.environ.copy()
        env['LD_LIBRARY_PATH']=':'.join(sorted({str(p.parent) for p in cuda_root.rglob('*.so*')})+[env.get('LD_LIBRARY_PATH','')])
        (work/'service.py').write_text(ASR_SERVICE,encoding='utf-8')
        asr_process=subprocess.Popen([sys.executable,'-u',str(work/'service.py')],env=env,
            stdin=subprocess.PIPE,stdout=subprocess.PIPE,text=True,encoding='utf-8')
    path=work/'audio.npy';np.save(path,samples)
    asr_process.stdin.write(json.dumps(dict(path=str(path),language=language,live=live))+'\n')
    asr_process.stdin.flush()
    # Bounded wait even if a CUDA subprocess crashes or hangs.
    import concurrent.futures
    pool=concurrent.futures.ThreadPoolExecutor(max_workers=1)
    pending=pool.submit(asr_process.stdout.readline)
    try: line=pending.result(timeout=900)
    except concurrent.futures.TimeoutError:
        release_asr();raise RuntimeError('ASR subprocess timed out')
    finally: pool.shutdown(wait=False)
    result=json.loads(line)
    if result.get('error'): raise RuntimeError('ASR failed; see notebook logs')
    return result


def generate_text(task):
    global llm_model, llm_tokenizer
    if torch.cuda.device_count() < 2:
        release_asr()
    llm_device = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'
    from transformers import AutoTokenizer, AutoModelForCausalLM
    if llm_model is None:
        print('Loading Qwen3-4B on GPU...',flush=True)
        llm_tokenizer=AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')
        llm_model=AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B',torch_dtype=torch.float16,
            device_map=llm_device,attn_implementation='sdpa').eval()
    messages=task['messages']
    assert isinstance(messages,list) and len(messages)<30
    schema=task.get('schema')
    if schema:
        messages=[dict(role='system',content='Return only valid JSON matching this schema: '+json.dumps(schema,ensure_ascii=False))]+messages
    prompt=llm_tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs=llm_tokenizer(prompt,return_tensors='pt').to(llm_device)
    maximum=min(int(task.get('max_tokens',800)),1500)
    if inputs.input_ids.shape[1]+maximum>8192: raise ValueError('LLM context exceeds 8192 tokens')
    with torch.inference_mode():
        output=llm_model.generate(**inputs,max_new_tokens=maximum,do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id)
    text=llm_tokenizer.decode(output[0,inputs.input_ids.shape[1]:],skip_special_tokens=True)
    return dict(job_id=task['job_id'],text=text,model='Qwen/Qwen3-4B')



# Persistent centroids associate sliding-window diarization labels across jobs.
live_pipeline = None
live_sessions = {}

def live_speakers(samples, words, session):
    global live_pipeline
    from huggingface_hub import snapshot_download
    from pyannote.audio import Pipeline
    if live_pipeline is None:
        folder = snapshot_download('pyannote/speaker-diarization-community-1',
            revision='3533c8cf8e369892e6b79ff1bf80f7b0286a54ee', token=HF_TOKEN)
        live_pipeline = Pipeline.from_pretrained(folder).to(torch.device('cuda:0'))
    waveform = torch.from_numpy(samples[None,:])
    output = live_pipeline({'waveform': waveform, 'sample_rate':16000})
    tracks = [(turn, label) for turn, _, label in output.exclusive_speaker_diarization.itertracks(yield_label=True)]
    if session not in live_sessions:
        if len(live_sessions) >= 4:
            live_sessions.pop(next(iter(live_sessions)))
        live_sessions[session] = []
    centroids = live_sessions[session]
    mapping = {}
    for label in {label for _, label in tracks}:
        spans = [turn for turn, name in tracks if name == label and turn.duration >= .8]
        if not spans:
            continue
        span = max(spans, key=lambda turn: turn.duration)
        audio = waveform[:,int(span.start*16000):int(span.end*16000)]
        if audio.shape[-1] < 12800:
            continue
        vector = np.asarray(live_pipeline._embedding(audio[None,:,:]),dtype=np.float32).reshape(-1)
        norm = np.linalg.norm(vector)
        if not np.isfinite(vector).all() or norm <= 0:
            continue
        vector /= norm
        scores = [float(np.dot(vector, ref)) for ref in centroids]
        best = int(np.argmax(scores)) if scores else -1
        if best >= 0 and scores[best] >= .60:
            merged = centroids[best]*.9 + vector*.1
            centroids[best] = merged / np.linalg.norm(merged)
        elif not scores or max(scores) < .40:
            if len(centroids) >= 8:
                continue
            best = len(centroids)
            centroids.append(vector)
        else:
            continue
        mapping[label] = 'speaker_' + str(best+1)
    rows = []
    for word in words:
        overlaps = [(max(0,min(word['end'],turn.end)-max(word['start'],turn.start)),label) for turn,label in tracks]
        match = max(overlaps, default=(0,None))
        speaker = mapping.get(match[1]) if match[0] > 0 else None
        rows.append(dict(start=word['start'],end=word['end'],text=word['word'],speaker=speaker))
    return rows

# Embedded into the private Kaggle worker, not executed locally.
refine_pipeline=None
refinement_ready=threading.Event()
refine_sessions={}
background_refinement=None
background_refinement_id=None


def prepare_refinement():
    global refine_pipeline
    if torch.cuda.device_count()<2:return
    from huggingface_hub import snapshot_download
    from pyannote.audio import Pipeline
    began=time.perf_counter()
    model_dir=snapshot_download('pyannote/speaker-diarization-community-1',
        revision='3533c8cf8e369892e6b79ff1bf80f7b0286a54ee',token=HF_TOKEN)
    refine_pipeline=Pipeline.from_pretrained(model_dir).to(torch.device('cuda:1'))
    with torch.inference_mode():
        refine_pipeline({'waveform':torch.zeros(1,160000),'sample_rate':16000})
        refine_pipeline._embedding(torch.zeros(1,1,48000))
    refinement_ready.set()
    # Announce immediately, instead of waiting for the next 20-second heartbeat.
    try:requests.post(WORKER_URL+'/heartbeat',headers=HEADERS,
        json={'features':['multilingual','llm','live_asr','live_speakers','live_refine','shared_turns']},timeout=15)
    except requests.RequestException:pass
    print(f'Speaker refinement ready on GPU 1 ({time.perf_counter()-began:.1f}s warm-up).',flush=True)


def warm_refinement():
    try:prepare_refinement()
    except Exception as exc:
        print('Speaker warm-up failed:',type(exc).__name__,flush=True)
        print('Captions remain available; speaker capability is not advertised.',flush=True)




def refine_speakers(samples, session, offset=0, units=None):
    """Never spread a local cluster ID over tracks without acoustic verification."""
    assert refinement_ready.is_set(), 'Wait for Speaker refinement ready before starting Live.'
    began=time.perf_counter()
    if session not in refine_sessions and len(refine_sessions)>=4:
        del refine_sessions[next(iter(refine_sessions))]
    memory=refine_sessions.setdefault(session,SpeakerMemory())
    with torch.inference_mode():
        output=refine_pipeline({'waveform':torch.from_numpy(samples[None,:]),'sample_rate':16000})
        annotation=getattr(output,'exclusive_speaker_diarization',None)
        if annotation is None:annotation=output.speaker_diarization
        tracks=[turn for turn,_,_ in annotation.itertracks(yield_label=True)]
        diarization_seconds=time.perf_counter()-began
        if units is not None:
            raw_tracks=[dict(start=float(t.start),end=float(t.end),speaker=label) for t,_,label in annotation.itertracks(yield_label=True)]
            labelled=refine_sentence_units(samples,units,raw_tracks,session,offset)
            return dict(tracks=labelled,timing=dict(diarization_seconds=round(diarization_seconds,3),
                total_seconds=round(time.perf_counter()-began,3)))
        observations=[]
        for turn in tracks:
            observation=None;purity=0.
            if turn.duration>=1.5:
                # Compare subwindows within a track before enrolling its identity.
                width=min(1.8,turn.duration)
                starts=sorted({turn.start,(turn.start+turn.end-width)/2,turn.end-width})
                vectors=[]
                for start in starts:
                    audio=torch.from_numpy(samples[round(start*16000):round((start+width)*16000)][None,None,:])
                    if audio.shape[-1]<24000:continue
                    vector=np.asarray(refine_pipeline._embedding(audio),dtype=np.float32).reshape(-1)
                    norm=np.linalg.norm(vector)
                    if np.isfinite(vector).all() and norm>0:vectors.append(vector/norm)
                if vectors:
                    purity=min(float(np.dot(a,b)) for a in vectors for b in vectors)
                    if purity>=.55:
                        vector=np.mean(vectors,axis=0)
                        observation=memory.observe(vector,offset+turn.start,offset+turn.end)
            observations.append((turn,observation,purity))
    result=[]
    for turn,observation,purity in observations:
        speaker=observation['speaker'] if observation else None
        result.append(dict(start=float(turn.start),end=float(turn.end),speaker=speaker,
            evidence=dict(policy='per-track-purity-v1',accepted=bool(speaker),
                within_track_similarity=round(purity,3),support_seconds=round(observation['seconds'],2) if observation else 0)))
    return dict(tracks=result,timing=dict(diarization_seconds=round(diarization_seconds,3),
        total_seconds=round(time.perf_counter()-began,3)))


def run_background_refinement(endpoint, raw):
    try:
        result=dispatch(raw)
        for attempt in range(4):
            try:
                response=requests.post(endpoint+'/result',headers=HEADERS,data=result,timeout=30)
                if response.status_code in (401,403,409,410):return
                response.raise_for_status();return
            except requests.RequestException:
                if stop.wait(2):return
    except Exception as exc:
        print('Background speaker refinement failed:',type(exc).__name__,flush=True)
        try:requests.post(endpoint+'/failed',headers=HEADERS,timeout=15)
        except requests.RequestException:pass


def dispatch(raw):
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        if 'task.json' not in z.namelist(): return process(raw)
        assert z.getinfo('task.json').file_size<=150000
        task=json.loads(z.read('task.json'))
        if task['kind']=='llm':
            assert set(z.namelist())=={'task.json'}
            result=generate_text(task)
        elif task['kind']=='live_refine':
            assert set(z.namelist())=={'task.json','audio.wav'}
            assert z.getinfo('audio.wav').file_size<16000*2*31
            import soundfile as sf
            samples,sr=sf.read(io.BytesIO(z.read('audio.wav')),dtype='float32')
            assert sr==16000 and samples.ndim==1 and np.isfinite(samples).all()
            assert isinstance(task['session'],str) and len(task['session'])<=80
            result=dict(job_id=task['job_id'],**refine_speakers(samples,task['session'],float(task.get('offset',0)),task.get('units')))
        elif task['kind']=='live_asr':
            assert set(z.namelist())=={'task.json','audio.wav'}
            assert z.getinfo('audio.wav').file_size<16000*2*15
            import soundfile as sf
            samples,sr=sf.read(io.BytesIO(z.read('audio.wav')),dtype='float32')
            assert sr==16000 and samples.ndim==1 and np.isfinite(samples).all()
            assert task['language'] in ('vi','en','auto')
            recognized=recognize(samples,task['language'],live=True)
            rows = recognized['segments']
            if task.get('session'):
                rows = live_speakers(samples,recognized['words'],task['session'])
            result=dict(job_id=task['job_id'],segments=rows,language=recognized['language'],
                asr_model=recognized['asr_model'],inference_seconds=recognized['inference_seconds'])
        else: raise ValueError('Unknown task')
    return json.dumps(result,ensure_ascii=False).encode('utf-8')

print('Speaker engine: audio-reference-v2.2 (batched audio windows + shared Live/Record boundaries)', flush=True)
print('Warming up multilingual live ASR...', flush=True)
recognize(np.zeros(16000,dtype=np.float32), 'vi', live=True)
thread = threading.Thread(target=heartbeat, daemon=True)
thread.start()
refinement_warmup=threading.Thread(target=warm_refinement,daemon=True)
refinement_warmup.start()
worker_http=requests.Session()
print('Worker running. Choose Community-1 in your local demo. Stop this cell to disconnect.')
try:
    while not stop.is_set():
        try:
            response = worker_http.get(WORKER_URL+'/next', headers=HEADERS,
                params={'inline':1,'wait':15,**({'exclude':background_refinement_id} if background_refinement is not None and background_refinement.is_alive() else {})},timeout=30)
            if response.status_code in (401, 403, 410):
                print('Connection expired. Reconnect from the demo.'); break
            response.raise_for_status()
            inline_audio=response.content if response.headers.get('X-VoxMeet-Job') else None
            job_id=response.headers.get('X-VoxMeet-Job') or (None if response.status_code==204 else response.json()['job'])
        except requests.RequestException:
            print('Connection interrupted; retrying in 10 seconds.')
            stop.wait(10); continue
        if not job_id:
            stop.wait(.05 if response.status_code==204 else .5); continue
        if background_refinement is not None and background_refinement.is_alive() and job_id==background_refinement_id:
            stop.wait(.1);continue
        endpoint = BASE + '/exchange/' + job_id
        try:
            if inline_audio is not None:
                raw=inline_audio
            else:
                response = worker_http.get(endpoint+'/audio', headers=HEADERS, timeout=180)
                response.raise_for_status()
                raw=response.content
            with zipfile.ZipFile(io.BytesIO(raw)) as archive:
                metadata=json.loads(archive.read('task.json')) if 'task.json' in archive.namelist() else {}
            if metadata.get('kind')=='live_refine':
                if background_refinement is not None and background_refinement.is_alive():
                    stop.wait(.1);continue
                background_refinement_id=job_id
                background_refinement=threading.Thread(target=run_background_refinement,args=(endpoint,raw),daemon=True)
                background_refinement.start();continue
            result = dispatch(raw)
        except Exception as exc:
            # Do not print request URLs containing connection identifiers.
            print('Processing failed:', type(exc).__name__)
            if not isinstance(exc, requests.RequestException): traceback.print_exc()
            try: worker_http.post(endpoint+'/failed', headers=HEADERS, timeout=20)
            except requests.RequestException: pass
            stop.wait(5); continue
        # Retry result transfer without rerunning inference. Server writes idempotently.
        for attempt in range(12):
            try:
                response = worker_http.post(endpoint+'/result', headers=HEADERS, data=result, timeout=120)
                if response.status_code in (401, 403, 409, 410):
                    print('Job closed; result remains in /kaggle/working.'); break
                response.raise_for_status()
                print('Transcript returned automatically. Waiting for the next recording.'); break
            except requests.RequestException:
                stop.wait(5)
        else:
            print('Could not return result. Stopping to avoid reprocessing this audio.'); break
finally:
    stop.set()
    worker_http.close()
    if background_refinement is not None:background_refinement.join(timeout=30)
    release_asr()
    release_llm()
    thread.join(timeout=20)
    print('Worker stopped. Audio and results in /kaggle/working are temporary until saved.')
